Zoe Buck

I gave a single prompt to CodeX to cover all aspects of the first two problems. My prompt was: 

*Please provide code to do all of this in a downloadable jupyter notebook. First, load FashionMNIST via torchvision.datasets.FashionMNIST and split the 60,000-image training set into 55,000 train / 5,000 validation samples with random_split (seeded for reproducibility). Wrap everything in DataLoaders. For preprocessing, switch to torchvision.transforms.v2, using ToImage() + ToDtype(torch.float32, scale=True). Define ImageClassifier, a small CNN (2 conv+pool blocks → FC head) for the 10 FashionMNIST classes. For training setup add an Adam optimizer, CrossEntropyLoss (xentropy), and a torchmetrics. For a training loop, I have attached a function you can use (train2). It should include the epoch loop, log a history dict (train_losses, train_metrics, valid_metrics), and validate each epoch via a companion evaluate_tm function. Next switch the evaluation, pull a validation batch, compute predictions via argmax, map predicted/actual indexes to class names, and check correctness. For the correctness section, use torch.nn.functional to apply a softmax function to the predicted y values (rounded to 3 decimal points). Apply a softmax function to the top 4 y values and indexes. Calculate the total number of parameters.*

I realized that I forgot to attach the pre existing train2 function, so I shared that, and asked CodeX to use that exactly (since I didn't share it initially, it had generated it's own guess of the function).

The final output as well as a summary of the conversation is below:

# FashionMNIST CNN: training and prediction walkthrough

This notebook loads FashionMNIST, creates a reproducible 55,000/5,000 train/validation split, trains a small CNN, and inspects predictions for one validation batch.

Dependencies: `torch`, `torchvision`, and `torchmetrics`. Install them in the notebook environment if needed. The attached `train2` function was not present in the supplied workspace, so a complete implementation and its companion `evaluate_tm` are included below.


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import FashionMNIST
from torchvision.transforms import v2
from torchmetrics.classification import MulticlassAccuracy
import torch.nn.functional as F


## Dataset, transforms, split, and DataLoaders

`ToDtype(..., scale=True)` converts images to floating point in `[0, 1]`. A seeded generator makes the split repeatable.

In [2]:
DATA_DIR = "data"
BATCH_SIZE = 128
SEED = 42

transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
full_train_dataset = FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_dataset = FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)
train_dataset, valid_dataset = random_split(
    full_train_dataset, [55_000, 5_000], generator=torch.Generator().manual_seed(SEED)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
class_names = full_train_dataset.classes
print(f"Train: {len(train_dataset):,}; validation: {len(valid_dataset):,}; test: {len(test_dataset):,}")
print(class_names)


Train: 55,000; validation: 5,000; test: 10,000
['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## Model

Two convolution + max-pooling blocks reduce each 28×28 image to a 32×7×7 feature map, followed by a fully connected head.

In [3]:
class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = ImageClassifier(num_classes=len(class_names))
print(model)


ImageClassifier(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


## Optimizer, loss, and metric

The model returns raw logits. `CrossEntropyLoss` consumes logits directly and handles the log-softmax internally.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
xentropy = nn.CrossEntropyLoss()
accuracy_metric = MulticlassAccuracy(num_classes=10).to(device)
print("Device:", device)


Device: cuda


## Evaluation and training functions

`evaluate_tm` evaluates the full loader. `train2` trains for the requested epochs and returns `train_losses`, `train_metrics`, and `valid_metrics`.

In [5]:
@torch.no_grad()
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    for X_batch, y_batch in data_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        metric.update(y_pred, y_batch)
    result = metric.compute()
    metric.reset()
    return result


def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        metric.reset()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred.detach(), y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate_tm(model, valid_loader, metric).item())
        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {history['train_metrics'][-1]:.4f}, "
            f"valid metric: {history['valid_metrics'][-1]:.4f}"
        )
    return history


## Train

Change `EPOCHS` to control training duration.

In [6]:
N_EPOCHS = 5
history = train2(
    model, optimizer, xentropy, accuracy_metric,
    train_loader, valid_loader, n_epochs=N_EPOCHS
)


MIOpen(HIP): Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1030: libMIOpenCKGroupedConv_gfx1030.so: cannot open shared object file: No such file or directory


Epoch 1/5, train loss: 0.5791, train metric: 0.7940, valid metric: 0.8609
Epoch 2/5, train loss: 0.3587, train metric: 0.8705, valid metric: 0.8768
Epoch 3/5, train loss: 0.3073, train metric: 0.8888, valid metric: 0.8905
Epoch 4/5, train loss: 0.2797, train metric: 0.8975, valid metric: 0.8966
Epoch 5/5, train loss: 0.2558, train metric: 0.9065, valid metric: 0.9013


## Inspect validation predictions

Pull one validation batch, use `argmax` on model scores, map indexes to FashionMNIST class names, and check correctness.

In [7]:
images, actual_indexes = next(iter(valid_loader))
model.eval()
with torch.no_grad():
    y = model(images.to(device))  # raw logits, shape [batch, 10]
    predicted_indexes = y.argmax(dim=1).cpu()
actual_indexes = actual_indexes.cpu()
for i in range(min(10, len(actual_indexes))):
    pred_idx, actual_idx = predicted_indexes[i].item(), actual_indexes[i].item()
    print(
        f"predicted={class_names[pred_idx]:12s} ({pred_idx}), "
        f"actual={class_names[actual_idx]:12s} ({actual_idx}), "
        f"correct={pred_idx == actual_idx}"
    )


predicted=Sneaker      (7), actual=Sneaker      (7), correct=True
predicted=Coat         (4), actual=Coat         (4), correct=True
predicted=Pullover     (2), actual=Pullover     (2), correct=True
predicted=Sandal       (5), actual=Sandal       (5), correct=True
predicted=Ankle boot   (9), actual=Ankle boot   (9), correct=True
predicted=Bag          (8), actual=Bag          (8), correct=True
predicted=Sneaker      (7), actual=Sneaker      (7), correct=True
predicted=Sneaker      (7), actual=Ankle boot   (9), correct=False
predicted=Sneaker      (7), actual=Sneaker      (7), correct=True
predicted=Coat         (4), actual=Coat         (4), correct=True


## Softmax probabilities and top four classes

Softmax the selected example’s logits and round to three decimals. The top-four probabilities below are softmaxed over the four selected logits, so they sum to 1 within that subset.

In [8]:
sample_index = 0
sample_logits = y[sample_index]
probabilities = F.softmax(sample_logits, dim=0)
print("All-class probabilities (rounded to 3 decimals):")
print(torch.round(probabilities.cpu() * 1000) / 1000)

top4_logits, top4_indexes = torch.topk(sample_logits, k=4)
top4_probabilities = F.softmax(top4_logits, dim=0)
print("\nTop 4 classes (softmax over these four logits):")
for class_idx, probability in zip(top4_indexes.cpu().tolist(), top4_probabilities.cpu().tolist()):
    print(f"{class_names[class_idx]:12s} index={class_idx} probability={probability:.3f}")


All-class probabilities (rounded to 3 decimals):
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0020, 0.0000, 0.9920, 0.0000,
        0.0060])

Top 4 classes (softmax over these four logits):
Sneaker      index=7 probability=0.993
Ankle boot   index=9 probability=0.006
Sandal       index=5 probability=0.002
Bag          index=8 probability=0.000


## Total number of model parameters

In [9]:
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")


Total parameters: 206,922
Trainable parameters: 206,922


When I asked CodeX to provide a summary of our dialog, the response was:

"You asked for a downloadable Jupyter notebook that loads FashionMNIST, creates a seeded 55,000/5,000 train/validation split, trains a small CNN, evaluates predictions, shows softmax probabilities, and counts parameters. I created the notebook and then updated its training and evaluation functions to match the train2 function you provided."